# IND320 Project Work, Part 4 - Machine Learning

GitHub: [IND320-kahaland](https://github.com/kahalandkh/IND320-kahaland) \
Streamlit: [IND320 Energy Project](https://ind320-kahaland-vqqy43wgawwsxmytoyb9ah.streamlit.app)

## 0. Setup and imports

In [1]:
from datetime import datetime, timezone
import inspect
import logging
import os
from pathlib import Path
import sys

# Set the working directory to the project root before importing Streamlit
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
sys.path.insert(0, str(project_root))

import certifi
from dotenv import load_dotenv
from matplotlib import pyplot as plt
import pandas as pd
from pymongo import MongoClient
from river import metrics, time_series
import streamlit as st
from streamlit.logger import set_log_level

set_log_level("ERROR")

load_dotenv()

from modules.entsoe_mongodb import load_weekly_cross_border
from modules.forecasting import combine_weekly_data
from modules.magasinstatistikk_data import load_magasinstatistikk_data
from modules.nb_exchange_rates import get_nb_exchange_rates, load_weekly_exchange_rates
from modules.weather import load_regional_hourly_weather, hourly_weather_to_weekly

## 1. Norges Bank Currency Exchange Rates

The `get_nb_exchange_rates()` function in `modules/nb_exchange_rates.py` downloads exchange rates from the [Norges Bank API](https://www.norges-bank.no/en/topics/Statistics/open-data/guide-data-warehouse/). It supports selecting currencies, date ranges and frequency, and returns a pandas DataFrame with the original metadata.

The example below downloads EUR/NOK and USD/NOK rates.

In [2]:
print(inspect.getsource(get_nb_exchange_rates))

@st.cache_data(ttl="24h")
def get_nb_exchange_rates(start_date, end_date, currencies=("EUR", "USD"), frequency="B"):
    """Download exchange rates against NOK from the Norges Bank API."""

    # Accept both a single currency ("EUR") and multiple currencies ("EUR", "USD")
    if isinstance(currencies, str):
        currencies = (currencies,)

    # Norges Bank uses '+' to separate currencies in the API URL
    currency_codes = "+".join(currencies)
    url = f"https://data.norges-bank.no/api/data/EXR/{frequency}.{currency_codes}.NOK.SP"

    params = {"startPeriod": str(start_date),
              "endPeriod": str(end_date),
              "format": "csv",
              "locale": "en"}

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Read the semicolon-separated CSV response directly into memory and preserve the API metadata
    df = pd.read_csv(StringIO(response.text), sep=";")

    if df.empty:
        raise ValueError("No exchange rate

In [3]:
rates = get_nb_exchange_rates("2025-12-22", "2025-12-31")
rates

,FREQ,Frequency,BASE_CUR,Base Currency,QUOTE_CUR,Quote Currency,TENOR,Tenor,DECIMALS,CALCULATED,UNIT_MULT,Unit Multiplier,COLLECTION,Collection Indicator,TIME_PERIOD,OBS_VALUE
0,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-22,10.1213
1,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-23,10.0535
2,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-29,10.0518
3,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-30,10.0587
4,B,Business,USD,US dollar,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-31,10.0791
5,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-22,11.8875
6,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-23,11.8490
7,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-29,11.8270
8,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-30,11.8260
9,B,Business,EUR,Euro,NOK,Norwegian krone,SP,Spot,4,False,0,Units,C,ECB concertation time 14:15 CET,2025-12-31,11.8430


The function returned exchange rates for both EUR and USD. There are no observations for 24 to 28 December, which covers the Christmas holidays and the weekend.

## 2. Timezone Investigation

I looked at how dates and timezones are handled in Magasinstatistikk, ENTSO-E, Open-Meteo and Norges Bank. I also checked how often observations are recorded and whether weekends and holidays affect data availability, since these differences affect how the datasets can be combined for forecasting.

### 2.1 Norges Bank

The API returns exchange rates with observation dates but no timezone aware timestamps. The metadata specifies a reference time of 14:15 CET, and this label also appears during summer time. According to the [Norges Bank FAQ](https://www.norges-bank.no/en/topics/statistics/exchange_rates/valutakursar-faq), daily rates are published around 16:00 CET on business days.

I used Business frequency, so there are gaps for weekends and other days without published rates. I kept the original dates and metadata without applying any timezone conversion. The gaps between observations need to be considered when combining exchange rates with the other datasets.

In [4]:
winter = get_nb_exchange_rates("2026-01-15", "2026-01-15", currencies="EUR")
summer = get_nb_exchange_rates("2026-06-01", "2026-06-01", currencies="EUR")

samples = pd.concat([winter, summer])
samples[["TIME_PERIOD", "BASE_CUR", "Collection Indicator"]]

,TIME_PERIOD,BASE_CUR,Collection Indicator
0,2026-01-15,EUR,ECB concertation time 14:15 CET
0,2026-06-01,EUR,ECB concertation time 14:15 CET


### 2.2 Magasinstatistikk

Magasinstatistikk provides weekly reservoir filling data, with `date_id` referring to Sundays, including those that fall on holidays such as Christmas. According to [NVE's documentation](https://www.nve.no/energi/analyser-og-statistikk/om-magasinstatistikken/), the weekly values are based on measurements from the end of Sunday or up to two days earlier. The data is normally published on Wednesdays at 13:00, but publication may be delayed around holidays.

The API does not include timezone offsets. The `next_publication_date` field shows 13:00 in both winter and summer, suggesting that the publication time follows Norwegian local time (CET/CEST), although this is not explicitly stated. I kept the weekly dates unchanged, since they are reference dates rather than exact measurement timestamps.

In [5]:
df_mag = load_magasinstatistikk_data()

# Compare winter, summer and Christmas observations
dates = pd.to_datetime(["2022-12-25", "2023-12-24", "2026-01-18", "2026-07-05"])
cols = ["date_id", "iso_year", "iso_week", "next_publication_date"]

df_mag.loc[df_mag["date_id"].isin(dates), cols].drop_duplicates().sort_values("date_id")

,date_id,iso_year,iso_week,next_publication_date
171,2022-12-25,2022,51,2023-01-04T13:00:00
390,2023-12-24,2023,51,2024-01-03T13:00:00
3444,2026-01-18,2026,3,2026-01-28T13:00:00
6364,2026-07-05,2026,27,2026-07-15T13:00:00


### 2.3 ENTSO-E

According to [ENTSO-E's documentation](https://transparencyplatform.zendesk.com/hc/en-us/articles/12786431986964-Response-Time-Zone), API responses always use UTC timestamps. In Part 2, I downloaded cross-border electricity flow data for 2024 and 2025, processed it using Spark and Cassandra, and stored the results in MongoDB. I had some problems with duplicate timestamps around daylight-saving changes when transferring the data from pandas to Spark, which I solved by explicitly preserving the UTC offset. The MongoDB data is not updated automatically, so comparisons with other datasets are limited to the dates available in the collection.

I used the existing MongoDB data to check the transition from summer to winter time on 26 October 2025. After interpreting the stored timestamps as UTC and converting them to `Europe/Oslo`, the hour between 02:00 and 03:00 appears twice, once with UTC+2 and once with UTC+1. The UTC timestamps are different, so the two occurrences can still be distinguished.

Unlike Norges Bank, ENTSO-E also reports data for weekends and holidays. The data includes both hourly and 15-minute observations. I already accounted for these different intervals when calculating transferred energy in Part 2. The different observation intervals and UTC timestamps need to be accounted for when combining this data with the weekly reservoir values.

In [6]:
client = MongoClient(os.environ["MONGODB_URI"], tlsCAFile=certifi.where())
collection = client["ind320_project"]["cross_border_flows"]

# Check timestamps around the October 2025 daylight saving transition
query = {"norway_area": "BZN|NO2",
         "foreign_area": "BZN|DK1",
         "direction": "in",
         "timestamp": {"$gte": datetime(2025, 10, 26, 0, tzinfo=timezone.utc),
                       "$lt": datetime(2025, 10, 26, 2, tzinfo=timezone.utc)}}

records = list(collection.find(query, {"_id": 0, "timestamp": 1}).sort("timestamp"))

timestamps = pd.to_datetime([row["timestamp"] for row in records], utc=True)

display(pd.DataFrame({"UTC": timestamps, "Norwegian time": timestamps.tz_convert("Europe/Oslo")}))

# Check whether stored observations are present during Christmas
for day in ["2025-12-24", "2025-12-25", "2025-12-26"]:
    start = pd.Timestamp(day, tz="Europe/Oslo").tz_convert("UTC")
    end = (pd.Timestamp(day, tz="Europe/Oslo") + pd.Timedelta(days=1)).tz_convert("UTC")

    count = collection.count_documents({"norway_area": "BZN|NO2",
                                        "foreign_area": "BZN|DK1",
                                        "direction": "in",
                                        "timestamp": {"$gte": start.to_pydatetime(), "$lt": end.to_pydatetime()}})
    print(f"{day}: {count} observations")
    
client.close()

,UTC,Norwegian time
0,2025-10-26 00:00:00+00:00,2025-10-26 02:00:00+02:00
1,2025-10-26 00:15:00+00:00,2025-10-26 02:15:00+02:00
2,2025-10-26 00:30:00+00:00,2025-10-26 02:30:00+02:00
3,2025-10-26 00:45:00+00:00,2025-10-26 02:45:00+02:00
4,2025-10-26 01:00:00+00:00,2025-10-26 02:00:00+01:00
5,2025-10-26 01:15:00+00:00,2025-10-26 02:15:00+01:00
6,2025-10-26 01:30:00+00:00,2025-10-26 02:30:00+01:00
7,2025-10-26 01:45:00+00:00,2025-10-26 02:45:00+01:00


2025-12-24: 96 observations
2025-12-25: 96 observations
2025-12-26: 96 observations


### 2.4 Open-Meteo

In Part 3, I used the [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) to download hourly ERA5 temperature and precipitation data from sampled locations within each region. I selected `Europe/Berlin` so that the requested dates follow local time rather than UTC. Berlin and Norway both use CET in winter and CEST in summer. As recommended in the API documentation, I used Unix timestamps to handle daylight saving changes. These represent seconds since 1 January 1970 UTC. In `regional_hourly_averages()`, I interpret these timestamps as UTC and then convert them to local time using pandas. This ensures that each observation gets the correct offset, including during daylight saving changes.

The example below uses data from 25-27 October 2025 for one location in NO3 to show how the daylight saving change is handled. On 26 October, there were 25 hourly observations, including two occurrences of 02:00, one with UTC+2 and the other with UTC+1. Open-Meteo also uses a fixed UTC offset when determining the requested date boundaries, which can leave out hours near midnight. In Part 3, I handled this by requesting one additional day before and after the selected dates, converting the timestamps to local time and then filtering to the dates the user selected.

The hourly data includes weekends and holidays. For longer periods, daily data could be an alternative if I only need weekly weather values. This would reduce the amount of data downloaded, although Open-Meteo's API limits would still need to be considered.

In [7]:
# Look at one location to investigate timestamps
weather = load_regional_hourly_weather("NO3", start_date="2025-10-25", end_date="2025-10-27", n_points=1)

# Inspect the switch from summer to winter time
oct_26 = weather.loc[weather["date"].dt.strftime("%Y-%m-%d") == "2025-10-26"]

print(f"Observations on 26 October 2025: {len(oct_26)}")
display(oct_26.loc[oct_26["date"].dt.hour == 2, ["date"]])

# Check hourly observations during holidays
christmas_weather = load_regional_hourly_weather("NO3", start_date="2025-12-23", end_date="2025-12-27", n_points=1)

counts = christmas_weather.groupby(christmas_weather["date"].dt.strftime("%Y-%m-%d")).size()
print(counts.loc[["2025-12-24", "2025-12-25", "2025-12-26"]])

Observations on 26 October 2025: 25


,date
25,2025-10-26 02:00:00+02:00
26,2025-10-26 02:00:00+01:00


date
2025-12-24    24
2025-12-25    24
2025-12-26    24
dtype: int64


## 3. Reservoir Filling Forecasting

In this section, I will use River to forecast weekly reservoir filling levels. I will compare different models, parameter settings, training periods and forecast horizons, and investigate whether external variables such as weather, exchange rates and electricity transfers improve the predictions.

I will evaluate the models using historical forecasts and use the results to choose default settings for Streamlit. The final forecasting function will also support predictions for future weeks.

### 3.1 Preparing weekly data

Before forecasting, I added functions to prepare the datasets at a weekly resolution, reusing the existing data loading functions where possible.

Magasinstatistikk already provides weekly reservoir filling values. For exchange rates, I calculate the mean of the available business day observations. Weather data is aggregated into weekly mean temperature and total precipitation, while electricity imports and exports are summed across the international connections for each Norwegian bidding zone.

I use Norwegian Monday–Sunday weeks, labelled by Sunday, to align the datasets. I chose weekly resolution to keep the original reservoir observations and preserve changes within each month. Monthly data would mean fewer observations, but also less detail. Weekly forecasting should also be manageable in terms of computation time.

In [ ]:
# Select the region for forecasting
region = "NO3"
area_number = int(region[2:])

# Reservoir filling for the selected electricity area
reservoir_weekly = (df_mag.loc[(df_mag["area_type"] == "EL") & (df_mag["area_number"] == area_number),
                               ["date_id", "reservoir_fill_ratio"]].set_index("date_id").sort_index())

# Weekly EUR/NOK and USD/NOK exchange rates
currency_weekly = load_weekly_exchange_rates(reservoir_weekly.index.min().strftime("%Y-%m-%d"),
                                             reservoir_weekly.index.max().strftime("%Y-%m-%d"))

# Weekly international electricity transfers
electricity_weekly = load_weekly_cross_border(f"BZN|{region}")

start_date = pd.Timestamp("2023-01-01")
last_sunday = reservoir_weekly.index.max()
end_date = last_sunday + pd.Timedelta(days=1)

hourly_blocks = []

# Load weather in yearly blocks
for year in range(start_date.year, end_date.year + 1):
    block_start = max(start_date, pd.Timestamp(year=year, month=1, day=1))
    block_end = min(end_date, pd.Timestamp(year=year, month=12, day=31))
    hourly = load_regional_hourly_weather(region, block_start.strftime("%Y-%m-%d"), block_end.strftime("%Y-%m-%d"),
                                          n_points=100, batch_size=100)
    local_dates = hourly["date"].dt.tz_convert("Europe/Oslo").dt.tz_localize(None)
    hourly = hourly.loc[(local_dates >= block_start) & (local_dates < block_end + pd.Timedelta(days=1))]
    hourly_blocks.append(hourly)

# Convert hourly weather to complete weekly observations
hourly_weather = pd.concat(hourly_blocks, ignore_index=True).sort_values("date")
weather_weekly = hourly_weather_to_weekly(hourly_weather).loc[:last_sunday]

# Combine the selected weekly datasets
weekly_data = combine_weekly_data(reservoir_weekly, sources={"currency": currency_weekly,
                                                             "weather": weather_weekly,
                                                             "electricity": electricity_weekly})

# Show the available historical coverage
coverage = pd.DataFrame({"First week": weekly_data.apply(lambda s: s.first_valid_index()),
                         "Last week": weekly_data.apply(lambda s: s.last_valid_index()),
                         "Available weeks": weekly_data.notna().sum()})
display(coverage)
display(weekly_data.tail())

,First week,Last week,Available weeks
reservoir_fill_ratio,1995-01-08,2026-11-15,1663
eur_nok,1999-01-10,2026-11-15,1454
usd_nok,1995-01-15,2026-11-15,1662
temperature_mean_c,2023-01-08,2026-11-15,202
precipitation_total_mm,2023-01-08,2026-11-15,202
imports_mwh,2024-01-07,2025-12-28,104
exports_mwh,2024-01-07,2025-12-28,104
net_imports_mwh,2024-01-07,2025-12-28,104


,reservoir_fill_ratio,eur_nok,usd_nok,temperature_mean_c,precipitation_total_mm,imports_mwh,exports_mwh,net_imports_mwh
date_id,,,,,,,,
2026-10-18,0.724940,10.6487,9.50360,6.706483,41.940,NaN,NaN,NaN
2026-10-25,0.717647,10.5093,9.37124,7.529061,11.063,NaN,NaN,NaN
2026-11-01,0.720149,10.3668,9.29372,7.873263,38.837,NaN,NaN,NaN
2026-11-08,0.718705,10.4111,9.28732,5.613929,15.558,NaN,NaN,NaN
2026-11-15,0.697278,10.5790,9.50230,1.988682,42.612,NaN,NaN,NaN


The datasets cover quite different periods. Reservoir filling and USD/NOK go back to 1995, while EUR/NOK starts in 1999. I limited the weather data to 2023 onwards to avoid downloading unnecessarily large amounts of hourly data from the API, and the electricity data I collected cover 2024–2025.

### 3.2 Forecasting models and evaluation

<div style="background-color:#EAF2F7; border-left:5px solid #3E7FA6; padding:12px 16px; border-radius:6px;">

<h2>AI usage</h2>
To be filled out...
</div>

## Project log

I started by looking into the Norges Bank API and made a function to download exchange rates for different currencies, dates and frequencies. I put the function in a separate module so I can reuse it in Streamlit later, and kept the original metadata since some of it was useful for the timezone investigation. I tested the function with EUR and USD rates over Christmas 2025 and confirmed that there were no observations during the holidays or weekend when using Business frequency.

I then looked at how dates and timezones are handled in the four datasets used in the project. For ENTSO-E and Open-Meteo, I reused my work from Parts 2 and 3 to check how the timestamps behave around the daylight-saving change in October 2025. Both examples showed the repeated local hour with the correct UTC offsets, so I didn't need to change the existing timestamp handling. I also realised that using the same timezone would not be enough to combine the datasets. Magasinstatistikk has weekly observations, Norges Bank provides rates on business days, and weather and electricity data are recorded much more frequently. I therefore decided to use weekly data for forecasting, with different aggregation methods depending on the variable.

I then prepared the datasets at a weekly resolution and combined them for NO3. For weather, I used 100 sampled locations and limited the downloads to 2023 onwards to keep the API requests manageable.

To be continued...